# Smoke test for `ehtim.plotting.interactive`

Open in VSCode, pick the **jax-ehtim** kernel (top-right corner of the notebook), and run cells one at a time. Plotly figures render inline in the cell output — interactive (hover, zoom, pan, double-click to reset) without writing any HTML to disk.

In [1]:
import ehtim as eh
from ehtim.plotting import interactive

obs = eh.obsdata.load_uvfits('../data/sample.uvfits')
sites = sorted({s for pair in zip(obs.data['t1'], obs.data['t2']) for s in pair})
print('sites in sample.uvfits:', sites)

Welcome to eht-imaging! v 1.3.0 

Loading uvfits:  ../data/sample.uvfits
POLREP_UVFITS: circ
Number of uvfits Correlation Products: 4
No NX table in uvfits!


sites in sample.uvfits: ['ALMA', 'LMT', 'PDB', 'PV', 'SMA', 'SMT', 'SPT']


## `plot_bl` — single baseline

Pass `site1` and `site2` for one specific baseline. No legend (only one trace).

In [2]:
interactive.plot_bl(obs, 'ALMA', 'LMT', 'amp', show=False)

## `plot_bl` — all baselines, click-to-highlight

Omit `site1`/`site2` to plot every baseline as a uniform-gray trace. The legend behaviour:

- **Single click** on a gray legend entry → paints that baseline a distinct colour.
- **Single click** on a coloured entry → hides it (plotly's default toggle).
- **Single click** on a hidden entry → restores it as coloured.
- **Double-click** anywhere in the legend → paints *all* baselines (overrides plotly's default isolate).
- **"Show all / reset"** button (top-left of the plot) → restores every baseline to visible + gray.

Use `interactive.display(fig)` instead of returning `fig` so the JS handler that drives single/double-click is injected into the notebook output.


In [3]:
fig = interactive.plot_bl(obs, field='amp', show=False)
interactive.display(fig)

Try other fields — `'phase'`, `'snr'`, `'uvdist'`. The `'u'`/`'v'`/`'uvdist'` axes auto-scale to **Gλ** (or **Mλ** for shorter arrays) so labels read e.g. `4.2 Gλ` instead of `4.2e9 λ`.

SNR-cut example: `interactive.plot_bl(obs, field='amp', snrcut=5, show=False)`.


## `plotall` — scatter any two visibility fields

Plotly counterpart of `obs.plotall(field1, field2, ...)`.

- **`tag_bl=True` (default):** one trace per baseline, toggleable via legend.
- **`tag_bl=False`:** all visibilities pooled into one trace — useful when you just want the cloud.

In [4]:
# UV coverage. conj=True overlays the Hermitian conjugate half (-u,-v).
# Click a baseline in the legend to highlight it in colour; click again to hide.
fig = interactive.plotall(obs, 'u', 'v', conj=True, show=False)
interactive.display(fig)

In [5]:
# Radplot: amplitude vs baseline length. Pooled mode — single trace, useful
# when you just want to see the overall distribution.
interactive.plotall(obs, 'uvdist', 'amp', tag_bl=False, show=False)

In [6]:
# Per-baseline radplot. Click in the legend to highlight one in colour.
fig = interactive.plotall(obs, 'uvdist', 'amp', tag_bl=True, show=False)
interactive.display(fig)

## `plot_gains` — needs a Caltable

`sample.uvfits` has no caltable attached, so we synthesise a tiny fake one. For a real demo, replace this with a Caltable from `self_cal(obs, im, caltable=True)`.

`pol='both'` draws R and L as separate traces per site — R uses circles, L uses squares, so they stay separable even within the same colour.

In [7]:
import numpy as np
from ehtim.caltable import Caltable

eht = eh.array.load_txt('../arrays/EHT2017.txt')
demo_sites = ['ALMA', 'LMT', 'SMA', 'SMT']
rng = np.random.default_rng(0)
times = np.linspace(0.0, 24.0, 20)
datadict = {
    s: np.array(
        [(t, 1.0 + 0.1*rng.standard_normal() + 0j,
              1.0 + 0.1*rng.standard_normal() + 0j) for t in times],
        dtype=[('time','f8'),('rscale','c16'),('lscale','c16')]
    ) for s in demo_sites
}
ct = Caltable(ra=0.0, dec=0.0, rf=230e9, bw=4e9, datadict=datadict,
              tarr=eht.tarr, source='demo', mjd=58000, timetype='GMST')

interactive.plot_gains(ct, sites='all', gain_type='amp', pol='both', show=False)

## `dashboard` — combined view

2×2 grid: image + a data-product panel + gains + D-terms. Use this to inspect a full reconstruction (or a model + synthetic obs + self-cal output) at a glance.

**Per-panel legends.** Each scatter panel has its own legend on the right (data/model, sites, D-terms), so toggling visibility in one panel does not affect the others.

**`data_product` dropdown.** The top-of-figure dropdown switches what panel 2 shows. Options ported from `comp_plots.py`:

| option | source |
|---|---|
| Amplitude vs uv-distance | `obs.unpack(['uvdist','amp'])` + model |
| Re(vis) vs uv-distance | `obs.unpack(['vis'])` + model |
| Phase vs uv-distance | `obs.unpack(['phase'])` + model |
| χ residual vs uv-distance | `(amp − model_amp) / sigma` |
| Closure phase vs time | `obs.cphase_tri(s1,s2,s3)` + model |
| Log closure amp vs time | `obs.camp_quad(..., ctype='logcamp')` + model |
| Closure phase vs triangle uv-area | `cphase_tri` + `obsh.uv_area_triangle` |
| Log closure amp vs quadrangle uv-area | `camp_quad` + `obsh.uv_area_quadrangle` |

Triangle and quadrangle default to the ones with the most data points; override with `triangle=(s1,s2,s3)` / `quadrangle=(s1,s2,s3,s4)`.

**`plotp=True`** mirrors `Image.display(plotp=True)`: turns the Stokes-I colorbar on (label `Jy/px`) and overlays EVPA ticks per pixel, gated by `pcut` (I > pcut · Imax) and `mcut` (|m| > mcut). With `vec_cfun=True` the ticks are sampled by |m| via a small side colorbar.


In [8]:
# Build a (Image, Obsdata, Caltable) triple end-to-end so the dashboard has
# real data to show. Takes ~10 s — self_cal does the heavy lifting.
from ehtim.calibrating import self_cal

im_dash = eh.image.load_txt('../models/avery_sgra_eofn.txt')
eht_dash = eh.array.load_txt('../arrays/EHT2017.txt')

# Synthesise small D-terms so the D-term panel isn't all-zero.
rng = np.random.default_rng(42)
n = len(eht_dash.tarr)
eht_dash.tarr['dr'] = rng.normal(0, 0.05, n) + 1j * rng.normal(0, 0.05, n)
eht_dash.tarr['dl'] = rng.normal(0, 0.05, n) + 1j * rng.normal(0, 0.05, n)

obs_dash = im_dash.observe(eht_dash, tint=30, tadv=600,
                            tstart=0, tstop=24, bw=4e9,
                            sgrscat=False, ampcal=False, phasecal=False)
ct_dash = self_cal.self_cal(obs_dash, im_dash, processes=-1,
                             ttype='direct', caltable=True)

fig_dash = interactive.dashboard(im_dash, obs_dash, ct_dash, show=False)
interactive.display(fig_dash)

Loading text image:  ../models/avery_sgra_eofn.txt
Generating empty observation file . . . 


Producing clean visibilities from image with nfft FT . . . 


Adding gain + phase errors to data and applying a priori calibration . . . 
   Applying gain corruption: ampcal-->False
   Applying atmospheric phase corruption: phasecal-->False
Adding thermal noise to data . . . 
No stations specified in self cal: defaulting to calibrating all stations!
Computing the Model Visibilities with direct Fourier Transform...
Producing clean visibilities from image with direct FT . . . 


Not Using Multiprocessing
Scan 000/107 : [                              ]0%

Scan 001/107 : [                              ]0%

...

Scan 104/107 : [----------------------------- ]97%

Scan 105/107 : [----------------------------- ]98%

Scan 106/107 : [----------------------------- ]99%


self_cal time: 16.451505 s


Producing clean visibilities from image with direct FT . . . 


## Verifying the Gλ / Mλ axis auto-scaling

`plot_bl`, `plotall`, and the `dashboard` panel-2 uv-axes pick a single scale across all traces:

- max baseline ≥ 1e9 λ → display in **Gλ**
- otherwise → display in **Mλ**

Errors share the same divisor; hover tooltips and axis titles both reflect the chosen unit.


In [9]:
fig_uv = interactive.plot_bl(obs, field='uvdist', show=False)
print('y-axis title:', fig_uv.layout.yaxis.title.text)
interactive.display(fig_uv)


y-axis title: $u-v$ Distance (Gλ)


In [10]:
# The plotall radplot and uv-coverage figures pick the same unit across both
# axes when both are uv-fields (the uv-coverage plot below should read 'Gλ'
# on x AND y, not e.g. Gλ on x and Mλ on y).
fig_uv2 = interactive.plotall(obs, 'u', 'v', conj=True, show=False)
print('x title:', fig_uv2.layout.xaxis.title.text)
print('y title:', fig_uv2.layout.yaxis.title.text)


x title: $u$ (Gλ)
y title: $v$ (Gλ)


## `Show all / reset` button + double-click colour-all

Click any single legend entry to paint that one baseline. To paint *all* baselines in one go, double-click anywhere in the legend. To go back to all-gray, click the **Show all / reset** button at the top-left of the plot.


In [11]:
fig_btn = interactive.plot_bl(obs, field='amp', show=False)
# Sanity check: the button is wired up via plotly's updatemenus.
menus = fig_btn.layout.updatemenus or ()
labels = [b.label for m in menus for b in (m.buttons or ())]
print('top-bar buttons:', labels)
interactive.display(fig_btn)


top-bar buttons: ['Show all / reset']


## `dashboard(plotp=True)` — polarization ticks + colorbar

Mirrors `Image.display(plotp=True)`. Turn on the Stokes-I colorbar (label `Jy/px`) and overlay EVPA tick segments at the EVPA angle, length ∝ |P|, gated by `pcut` / `mcut`. With `vec_cfun=True` the tick midpoints are sampled by |m| via a small side colorbar.


In [12]:
# Quick polarized image so the ticks have something to render. Build it
# locally (independent of im_dash above, which is Stokes-only).
im_pol = eh.image.make_empty(64, 200 * eh.RADPERUAS, 17.761, -29.0, rf=230e9)
im_pol = im_pol.add_gauss(1.0, (60 * eh.RADPERUAS, 60 * eh.RADPERUAS, 0, 0, 0))
im_pol.add_qu(0.20 * im_pol.imarr(), 0.10 * im_pol.imarr())
im_pol.add_v(0.02 * im_pol.imarr())

obs_pol = im_pol.observe(eht_dash, tint=30, tadv=600,
                          tstart=0, tstop=24, bw=4e9,
                          sgrscat=False, ampcal=False, phasecal=False)

fig_pol = interactive.dashboard(
    im_pol, obs_pol, ct_dash,
    plotp=True, nvec=18, pcut=0.05, mcut=0.01, vec_cfun=True,
    show=False,
)
interactive.display(fig_pol)


Generating empty observation file . . . 


Producing clean visibilities from image with nfft FT . . . 
Adding gain + phase errors to data and applying a priori calibration . . . 
   Applying gain corruption: ampcal-->False
   Applying atmospheric phase corruption: phasecal-->False
Adding thermal noise to data . . . 


Producing clean visibilities from image with direct FT . . . 


## `dashboard` data-product dropdown

Panel 2 hosts a dropdown — switch between amp / vis / phase / χ-residual / cphase / log-camp options without rebuilding the figure. The closure-phase and closure-amplitude options need a triangle and quadrangle; pass them explicitly, or let the dashboard auto-pick the ones with the most samples.


In [13]:
# Auto-picked triangle + quadrangle. Print which ones the dashboard chose:
from ehtim.plotting.interactive import _pick_default_triangle, _pick_default_quad
print('default triangle:  ', _pick_default_triangle(obs_dash))
print('default quadrangle:', _pick_default_quad(obs_dash))

# Re-render the dashboard defaulting to the chisq-residual panel so the
# data-product dropdown is visibly populated; switch via the dropdown.
fig_dp = interactive.dashboard(
    im_dash, obs_dash, ct_dash,
    default_product='chisq_vs_uvdist',
    show=False,
)
interactive.display(fig_dp)


default triangle:   ('ALMA', 'APEX', 'SPT')


default quadrangle: ('ALMA', 'APEX', 'LMT', 'SPT')


Producing clean visibilities from image with direct FT . . . 


In [14]:
# Pin a specific triangle + quadrangle (skip the auto-pick) — useful when
# you want to compare specific closure quantities across runs.
sites_dash = sorted({s for pair in zip(obs_dash.data['t1'], obs_dash.data['t2'])
                     for s in pair})
print('available sites:', sites_dash)
# Override with whichever 3 / 4 sites you care about; this default just
# uses the auto-pick so the cell runs without manual editing.
tri = _pick_default_triangle(obs_dash)
quad = _pick_default_quad(obs_dash)
fig_dp2 = interactive.dashboard(
    im_dash, obs_dash, ct_dash,
    triangle=tri, quadrangle=quad,
    default_product='cphase_vs_time',
    show=False,
)
interactive.display(fig_dp2)


available sites: ['ALMA', 'APEX', 'JCMT', 'LMT', 'PV', 'SMA', 'SMT', 'SPT']


Producing clean visibilities from image with direct FT . . . 


## `obs_helpers.uv_area_triangle` / `uv_area_quadrangle`

These power the new "vs triangle / quadrangle area" data-product panels and are also available standalone for the matplotlib plotters.

- Triangle uv-area: `A = (1/2) · |u1·v2 − u2·v1|` from two of its three baseline vectors.
- Quadrangle uv-area: sum of the two triangles formed by three independent baseline vectors.


In [15]:
from ehtim.observing import obs_helpers as obsh
import numpy as np

# Right triangle with legs along u and v: area = 1/2 * base * height = 1.
assert obsh.uv_area_triangle(1.0, 0.0, 0.0, 2.0) == 1.0

# Vectorised over multiple baselines (matches the cphase_tri / camp_quad shape).
u1, v1 = np.array([1.0, 2.0]), np.array([0.0, 0.0])
u2, v2 = np.array([0.0, 0.0]), np.array([2.0, 3.0])
print('triangle areas:', obsh.uv_area_triangle(u1, v1, u2, v2))

# Unit square with corners (0,0), (1,0), (1,1), (0,1): baselines (1,0), (0,1), (-1,0).
print('unit square uv-area:', obsh.uv_area_quadrangle(1.0, 0.0, 0.0, 1.0, -1.0, 0.0))


triangle areas: [1. 3.]
unit square uv-area: 1.0
